In [0]:
# ============================================
# CityFlow — 04_streaming_openaq
# ============================================
import urllib.parse
import requests
from datetime import datetime

# Secure credentials
ACCESS_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-access-key")
SECRET_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="aws-secret-key")
OPENAQ_KEY = dbutils.secrets.get(scope="cityflow-secrets", key="openaq-api-key")
encoded_secret = urllib.parse.quote(SECRET_KEY, safe="")

S3_BUCKET = "cityflow-taxi-data"
BRONZE_PATH = f"s3a://{ACCESS_KEY}:{encoded_secret}@{S3_BUCKET}/bronze"

print("Credentials loaded!")

Credentials loaded!


In [0]:
def fetch_airquality():
    resp = requests.get(
        "https://air-quality-api.open-meteo.com/v1/air-quality"
        "?latitude=40.7128&longitude=-74.0060"
        "&hourly=pm10,pm2_5,nitrogen_dioxide,ozone"
        "&timezone=America/New_York&forecast_days=1"
    )
    data = resp.json()
    hourly = data["hourly"]
    times = hourly["time"]

    rows = [{"city": "New York",
             "measured_at": times[i],
             "pm2_5": float(hourly["pm2_5"][i] or 0),
             "pm10": float(hourly["pm10"][i] or 0),
             "nitrogen_dioxide": float(hourly["nitrogen_dioxide"][i] or 0),
             "ozone": float(hourly["ozone"][i] or 0),
             "ingestion_time": str(datetime.utcnow())}
            for i in range(len(times))]

    df = spark.createDataFrame(rows)
    df.write.format("delta").mode("append") \
        .save(f"{BRONZE_PATH}/airquality/")
    
    print(f" Fetched {len(rows)} rows at {datetime.utcnow()}")
    df.show(5)

fetch_airquality()
print("Air Quality Bronze saved!")

/home/spark-b3abdb42-fe16-4919-8185-64/.ipykernel/12671/command-6389170024801187-2860853235:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ingestion_time": str(datetime.utcnow())}
/home/spark-b3abdb42-fe16-4919-8185-64/.ipykernel/12671/command-6389170024801187-2860853235:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f" Fetched {len(rows)} rows at {datetime.utcnow()}")


 Fetched 24 rows at 2026-06-09 03:18:54.809638
+--------+--------------------+----------------+----------------+-----+----+-----+
|    city|      ingestion_time|     measured_at|nitrogen_dioxide|ozone|pm10|pm2_5|
+--------+--------------------+----------------+----------------+-----+----+-----+
|New York|2026-06-09 03:18:...|2026-06-08T00:00|             9.5| 67.0| 6.5|  6.4|
|New York|2026-06-09 03:18:...|2026-06-08T01:00|             7.7| 68.0| 9.3|  9.1|
|New York|2026-06-09 03:18:...|2026-06-08T02:00|             6.6| 67.0|11.4| 11.2|
|New York|2026-06-09 03:18:...|2026-06-08T03:00|             7.3| 62.0|12.3| 12.0|
|New York|2026-06-09 03:18:...|2026-06-08T04:00|             8.9| 56.0|12.2| 11.9|
+--------+--------------------+----------------+----------------+-----+----+-----+
only showing top 5 rows
Air Quality Bronze saved!
